<a href="https://colab.research.google.com/github/SauHin/Pneumonia-Detection-VGG16/blob/main/PneuScanAI_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Jika model ada di Drive
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 142.8 MB/s eta 0:00:00


In [ ]:
# Path sesuai lokasi model di drive
path_model_di_drive = '/content/drive/MyDrive/AOL_AI/model_pneumonia_vgg16.h5'

if os.path.exists(path_model_di_drive):
    print(f"✅ Model ditemukan: {path_model_di_drive}")
else:
    print("❌ Model TIDAK ditemukan!")

✅ Model ditemukan: /content/drive/MyDrive/AOL_AI/model_pneumonia_vgg16.h5
Siap lanjut ke pembuatan aplikasi!


In [ ]:
%%writefile app.py
"""
PneumoScan AI - Pneumonia Detection Web App
-------------------------------------------
Author: Jonathan Davin, Bryan Dale, Michael Hendrilie
Base Concept: Adapted from GeeksforGeeks tutorials on Image Classification.
Enhancements:
- Implemented VGG16 Transfer Learning architecture.
- Added Robust Preprocessing for Grayscale X-Rays.
"""

import streamlit as st
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import pandas as pd
import time
import os


st.set_page_config(
    page_title="PneumoScan AI",
    page_icon="🫁",
    layout="wide"
)

st.markdown("""
    <style>
    .stButton>button { width: 100%; background-color: #ff4b4b; color: white; }
    </style>
    """, unsafe_allow_html=True)

with st.sidebar:
    st.title("🫁 PneumoScan AI")
    st.info("Sistem Deteksi Pneumonia berbasis VGG16 Deep Learning.")

    st.info("ℹ️ **Model Info:**\n\nArchitecture: VGG16\n\nInput Size: 224x224 px\n\nDataset: Chest X-Ray Images")

    st.warning("⚠️ **Disclaimer:**\nAplikasi ini adalah dibuat dengan tujuan edukasi. Hasil prediksi AI BUKAN diagnosis medis final. Selalu konsultasikan dengan dokter profesional.")

# LOAD MODEL
@st.cache_resource
def load_ai_model():
    # Path file model
    filename = '/content/model_pneumonia_vgg16.h5'

    # Apakah file ada di folder yang sama dengan app.py (Lokal/GitHub)
    if os.path.exists(filename):
        return load_model(filename)

    # Cek path Google Drive
    drive_path = '/content/drive/MyDrive/' + filename
    if os.path.exists(drive_path):
        return load_model(drive_path)

    # Jika tidak ketemu di mana-mana
    return None


model = load_ai_model()

if model is None:
    st.error("❌ File Model tidak ditemukan!")
    st.stop()

# MAIN
st.write("# 🔬 Analisis X-Ray Paru-Paru")
st.write("Upload gambar X-Ray dada untuk mendeteksi indikasi Pneumonia.")

# Input File
uploaded_file = st.file_uploader("Upload X-Ray (JPG/PNG)", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    col1, col2 = st.columns([1, 1])

    with col1:
        st.subheader("🖼️ X-Ray Pasien")
        img_pil = Image.open(uploaded_file).convert('RGB')
        st.image(img_pil, use_container_width=True, caption="Original Image")

        # DEBUG VIEW
        with st.expander("🔍 Data Preprocessing"):
            st.write("Gambar ini akan di-resize menjadi **224x224** pixel dan dinormalisasi (nilai pixel dibagi 255).")
            img_resized = img_pil.resize((224, 224))
            img_array = image.img_to_array(img_resized)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = img_array / 255.0

            st.write(f"Tensor Shape: `{img_array.shape}`")
            st.write("Min Pixel Value:", np.min(img_array))
            st.write("Max Pixel Value:", np.max(img_array))

    with col2:
        st.subheader("📊 Hasil Analisis")

        analyze_button = st.button("Jalankan Diagnosis AI 🚀")

        if analyze_button:
            with st.spinner('Sedang memindai pola infeksi...'):
                time.sleep(1)

                # Prediksi
                try:
                    prediction = model.predict(img_array)
                    prob_normal = prediction[0][0]
                    prob_pneumonia = prediction[0][1]

                    # Logika Hasil
                    labels = ['Normal', 'Pneumonia']
                    probs = [prob_normal, prob_pneumonia]

                    # CHART PROBABILITAS
                    chart_data = pd.DataFrame(
                        probs,
                        index=labels,
                        columns=['Probability']
                    )

                    # Tampilan Hasil
                    if prob_pneumonia > prob_normal:
                        st.error("⚠️ TERDETEKSI PNEUMONIA")
                        confidence_score = prob_pneumonia
                    else:
                        st.success("✅ PARU-PARU NORMAL")
                        confidence_score = prob_normal

                    # Tingkat Keyakinan
                    st.metric(label="Tingkat Keyakinan (Confidence)", value=f"{confidence_score*100:.2f}%")

                    # Bar Chart Probabilitas
                    st.write("Distribusi Probabilitas:")
                    st.bar_chart(chart_data)

                    # Penjelasan Tambahan
                    if prob_pneumonia > 0.5:
                        st.info("💡 **Saran:** Segera konsultasikan dengan dokter spesialis paru untuk pemeriksaan lebih lanjut.")
                    else:
                        st.info("💡 **Saran:** Tidak ditemukan tanda-tanda infeksi signifikan, namun tetap jaga kesehatan.")

                except Exception as e:
                    st.error(f"Terjadi error sistem: {e}")

else:
    st.info("👈 Silakan upload gambar untuk memulai.")

Overwriting app.py


In [ ]:
# Library untuk Frontend & Tunneling
!pip install -q streamlit pyngrok tensorflow pillow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.0 MB/s eta 0:00:00


In [ ]:
import getpass
from pyngrok import ngrok

# Input token Ngrok
print("Masukkan Authtoken Ngrok Anda (Copy dari dashboard ngrok.com):")
token = getpass.getpass()

ngrok.set_auth_token(token)
ngrok.kill()

public_url = ngrok.connect(8501).public_url
print(f"\n🚀 APLIKASI AKTIF! Klik link ini: {public_url}")

!streamlit run app.py &>/content/logs.txt &

Masukkan Authtoken Ngrok Anda (Copy dari dashboard ngrok.com):
··········

🚀 APLIKASI AKTIF! Klik link ini: https://ascitical-breann-commemorational.ngrok-free.dev
